# Frozen candidate CPU validation analysis

Run from top to bottom after all 20 new runs are complete. Repository synchronization happens before project imports; this notebook contains no training commands and never loads held-out test data.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

CONTENT_ROOT = Path('/content')
REPO_URL = 'https://github.com/khyitty/VitalDB-Feature-Selection.git'
REPO_DIR = CONTENT_ROOT / 'VitalDB-Feature-Selection'
REQUIRED_WORKFLOW_COMMIT = '8df2802'
ROOT = Path('/content/drive/MyDrive/VitalDB-Feature-Selection')
DATASET_DIR = ROOT / 'data/modeling/full'
SELECTION_DIR = ROOT / 'outputs/predictive_feature_selection_30s'
RETRAINING_ROOT = ROOT / 'outputs/frozen_candidate_retraining_validation_only'
REGISTRY = RETRAINING_ROOT / 'candidate_source_registry.json'
OUTPUT_DIR = RETRAINING_ROOT / 'analysis'

In [ ]:
import importlib
import os
import shlex
import shutil
import subprocess
import sys

def run_command(command, *, cwd=None, check=True, stream=False):
    command = [str(part) for part in command]
    print('$', shlex.join(command))
    result = subprocess.run(
        command, cwd=cwd, check=False, capture_output=not stream, text=True
    )
    if result.stdout:
        print('[stdout]')
        print(result.stdout.rstrip())
    if result.stderr:
        print('[stderr]')
        print(result.stderr.rstrip())
    if check and result.returncode != 0:
        raise RuntimeError(
            f'Command failed with exit status {result.returncode}: {shlex.join(command)}\n'
            f'stdout:\n{result.stdout or "<empty>"}\n'
            f'stderr:\n{result.stderr or "<empty>"}'
        )
    return result

def verify_repository_state(repo_dir, required_ancestor):
    local_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'HEAD']).stdout.strip()
    origin_head = run_command(['git', '-C', repo_dir, 'rev-parse', 'origin/main']).stdout.strip()
    remote_url = run_command(['git', '-C', repo_dir, 'remote', 'get-url', 'origin']).stdout.strip()
    branch_result = run_command(['git', '-C', repo_dir, 'branch', '--show-current'], check=False)
    branch = branch_result.stdout.strip() or '<detached>'
    if local_head != origin_head:
        raise RuntimeError(f'Repository is not synchronized: HEAD={local_head}, origin/main={origin_head}.')
    ancestor = run_command(
        ['git', '-C', repo_dir, 'merge-base', '--is-ancestor', required_ancestor, 'HEAD'],
        check=False,
    )
    if ancestor.returncode != 0:
        raise RuntimeError(
            f'Required workflow commit {required_ancestor} is not an ancestor of HEAD {local_head}. '
            f'Git stderr: {ancestor.stderr or "<empty>"}'
        )
    recent = run_command(['git', '-C', repo_dir, 'log', '-3', '--oneline']).stdout.strip()
    print('Repository state:', {'HEAD': local_head, 'origin/main': origin_head, 'origin': remote_url, 'branch': branch})
    print('Recent commits:\n' + recent)
    return {'head': local_head, 'origin_main': origin_head, 'origin_url': remote_url, 'branch': branch}

def synchronize_repository(repo_dir, repo_url, required_ancestor, *, allowed_parent):
    repo_dir = Path(repo_dir)
    allowed_parent = Path(allowed_parent).resolve()
    if repo_dir.parent.resolve() != allowed_parent:
        raise RuntimeError(f'Refusing repository operation outside {allowed_parent}: {repo_dir}')
    allowed_parent.mkdir(parents=True, exist_ok=True)
    os.chdir(allowed_parent)
    git_dir = repo_dir / '.git'
    if git_dir.is_dir():
        current_remote = run_command(
            ['git', '-C', repo_dir, 'remote', 'get-url', 'origin'], check=False
        )
        if current_remote.returncode != 0:
            run_command(['git', '-C', repo_dir, 'remote', 'add', 'origin', repo_url])
        elif current_remote.stdout.strip() != repo_url:
            print(f'Updating origin URL: {current_remote.stdout.strip()} -> {repo_url}')
            run_command(['git', '-C', repo_dir, 'remote', 'set-url', 'origin', repo_url])
        run_command(['git', '-C', repo_dir, 'fetch', 'origin', 'main', '--prune'])
        run_command(['git', '-C', repo_dir, 'reset', '--hard', 'origin/main'])
    else:
        if repo_dir.exists() or repo_dir.is_symlink():
            print('Replacing non-Git temporary repository path:', repo_dir)
            if repo_dir.is_symlink():
                repo_dir.unlink()
            else:
                shutil.rmtree(repo_dir)
        run_command(['git', 'clone', repo_url, repo_dir])
    return verify_repository_state(repo_dir, required_ancestor)

def prepare_project_imports(repo_dir, prefixes=('src', 'scripts')):
    stale = [
        name for name in list(sys.modules)
        if any(name == prefix or name.startswith(prefix + '.') for prefix in prefixes)
    ]
    if stale:
        print('Purging stale project modules before re-import:', sorted(stale))
        for name in stale:
            sys.modules.pop(name, None)
    resolved_repo = Path(repo_dir).resolve()
    retained = []
    for entry in sys.path:
        try:
            if Path(entry or os.curdir).resolve() == resolved_repo:
                continue
        except OSError:
            pass
        retained.append(entry)
    sys.path[:] = [str(resolved_repo), *retained]
    importlib.invalidate_caches()
    return stale

REPOSITORY_STATE = synchronize_repository(
    REPO_DIR, REPO_URL, REQUIRED_WORKFLOW_COMMIT, allowed_parent=CONTENT_ROOT
)
PURGED_PROJECT_MODULES = prepare_project_imports(REPO_DIR)
os.chdir(REPO_DIR)

In [ ]:
run_command(
    [sys.executable, 'scripts/install_colab_dependencies.py', '--requirements', 'requirements-colab.txt'],
    cwd=REPO_DIR,
)

In [ ]:
import importlib
import pandas as pd
from IPython.display import Image, display

state = verify_repository_state(REPO_DIR, REQUIRED_WORKFLOW_COMMIT)
analysis_module = importlib.import_module('src.frozen_candidate_analysis')
module_path = Path(analysis_module.__file__).resolve()
if not module_path.is_relative_to(REPO_DIR.resolve()):
    raise RuntimeError(f'Stale analysis module loaded from {module_path}. Restart runtime.')
run_command([
    sys.executable, 'scripts/analyze_frozen_candidates.py',
    '--registry', REGISTRY,
    '--candidate-subsets', SELECTION_DIR / 'candidate_subsets.json',
    '--dataset-dir', DATASET_DIR,
    '--output-dir', OUTPUT_DIR,
    '--bootstrap-replicates', '10000',
    '--bootstrap-seed', '20260717',
], cwd=REPO_DIR)
for name in (
    'candidate_validation_aggregate.csv', 'paired_candidate_contrasts.csv',
    'paired_model_contrasts.csv', 'hierarchical_bootstrap_candidate_contrasts.csv',
    'hierarchical_bootstrap_model_contrasts.csv', 'candidate_pareto.csv',
    'candidate_evidence_table.csv',
):
    print(name)
    display(pd.read_csv(OUTPUT_DIR / name))
for path in sorted((OUTPUT_DIR / 'figures').glob('*.png')):
    print(path.name)
    display(Image(filename=str(path)))
print((OUTPUT_DIR / 'frozen_candidate_validation_report.md').read_text(encoding='utf-8')[:15000])